# Notebook 08 — Molecular Clocks

**Module:** 06 — Genetics and Evolution  
**Notebook:** 08 of 12  
**Prerequisites:** NB06 (distance methods), NB07 (character methods), NB03 (mutation rates)  
**Time estimate:** 75 minutes

> The molecular clock is the foundation of molecular dating — converting genetic
> distance into absolute time using fossil calibrations. It appears in every paper
> that estimates when two lineages diverged.

---
## Step 1 — Motivation

Fossils give us hard minimum ages for divergence events, but the fossil record is
incomplete. The molecular clock hypothesis — that DNA accumulates mutations at a
roughly constant rate — lets us use the amount of sequence divergence as a proxy
for time. Calibrating the clock with fossil dates and then projecting to nodes
without fossils is molecular dating, used in every dated phylogeny paper.

---
## Step 3 — Biological Background

### The Molecular Clock Hypothesis (Zuckerkandl & Pauling 1965)

Proteins (and later DNA) accumulate substitutions at an approximately constant rate
per year, independent of generation time and body size (for synonymous substitutions).

**Strict clock:** all lineages evolve at exactly the same rate per unit time.
**Relaxed clock:** rates vary across lineages but are modelled explicitly.

### Rate Variation Across Lineages

The strict clock fails for most real data:
- **Generation time effect:** organisms with shorter generations accumulate more
  mutations per year (rats evolve faster than elephants)
- **Metabolic rate:** higher metabolic rates → more reactive oxygen species → more
  oxidative DNA damage
- **Ancestral population size:** affects fixation time of new mutations

**Relative rate test (Sarich & Wilson 1973):** uses an outgroup to test if two
ingroup lineages evolve at the same rate. If d(A,outgroup) ≠ d(B,outgroup) for a
rooted comparison, the clock is violated between A and B.

### Calibration Strategies

| Source | Type | Uncertainty |
|--------|------|-------------|
| Fossil dates | Hard minimum, soft maximum | Preservational bias |
| Biogeographic events | Node age constraint | Geological dating uncertainty |
| Virus evolutionary rates | Known generation time | Suitable only for fast-evolving loci |
| Ancient DNA | Direct dated sequences | Degradation artifacts |

### Tools

- **BEAST2:** Bayesian relaxed-clock dating with fossil calibrations → posterior
  distribution of node ages
- **MCMCtree (PAML):** alternative Bayesian dating
- **r8s:** penalised likelihood for large datasets

---
## Step 4 — Mathematical Explanation

**Basic molecular clock equation:**
$$d = r \cdot t$$
where d = genetic distance, r = substitution rate (per site per year), t = divergence time.

**Estimating rate from calibrated node:**
$$r = \frac{d_{\text{calibrated}}}{2 \cdot t_{\text{fossil}}}$$
The factor of 2 accounts for independent evolution in both lineages since divergence.

**Relative rate test:**
For taxa A, B, outgroup C, if strict clock holds:
$$d(A, C) = d(B, C) \pm \text{stochastic error}$$
Test with: z = (d_{AC} − d_{BC}) / √Var(d_{AC} − d_{BC})

**Confidence interval on divergence time:**
From Poisson approximation, the variance of the number of substitutions is
proportional to the expected number: Var(S) ≈ E[S] = r·t·L.
Large L (long sequences) → narrow confidence intervals on t.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Strict molecular clock: estimate rate and date node
def molecular_clock_date(d: float, r: float) -> float:
    """Estimate divergence time from genetic distance and substitution rate.
    t = d / (2 * r)  [factor 2: both lineages evolved]"""
    return d / (2 * r)

def estimate_rate(d_calibrated: float, t_fossil: float) -> float:
    """Estimate substitution rate from calibrated node."""
    return d_calibrated / (2 * t_fossil)

# Primate molecular clock example
# Human–chimp JC distance ≈ 0.0101 (cytochrome b)
# Fossil calibration: human–chimp split ≈ 6 My
d_hc = 0.0101
t_fossil_hc_My = 6.0

r_cytb = estimate_rate(d_hc, t_fossil_hc_My)
print(f"Cytochrome b substitution rate: {r_cytb:.6f} substitutions/site/My")
print(f"(≈ {r_cytb * 1e6:.2f} × 10⁻⁶ per site per year)")

# Use rate to date human–gorilla split
d_hg = 0.0155
t_hg = molecular_clock_date(d_hg, r_cytb)
print(f"\nHuman–gorilla split estimate: {t_hg:.1f} My (expected: ~9 My)")

# Human–macaque
d_hm = 0.0830
t_hm = molecular_clock_date(d_hm, r_cytb)
print(f"Human–macaque split estimate:  {t_hm:.1f} My (expected: ~25–30 My)")

In [ ]:
# Cell 6.2 — Relative rate test
def relative_rate_test(d_AC: float, d_BC: float, d_AB: float) -> dict:
    """
    Test whether taxa A and B evolve at equal rates (strict clock test).
    C is the outgroup. Uses d_AC = d(A,outgroup), d_BC = d(B,outgroup).

    Under strict clock: d_AC should equal d_BC.
    Returns z-score and whether clock is rejected (|z| > 1.96, p < 0.05).
    """
    # Rate estimates: each lineage's branch length from ancestor to tip
    b_A = (d_AC + d_AB - d_BC) / 2  # branch length to A
    b_B = (d_BC + d_AB - d_AC) / 2  # branch length to B

    rate_diff = b_A - b_B

    # Variance of rate difference (Poisson approximation)
    # Var(d_AC) ≈ d_AC/L (where L is alignment length)
    # We can't estimate without L, so return raw difference and note this
    return {
        'branch_A': b_A,
        'branch_B': b_B,
        'rate_difference': rate_diff,
        'relative_rate_ratio': b_A / b_B if b_B > 0 else np.inf,
        'clock_violated': abs(b_A - b_B) > 0.1 * max(b_A, b_B),  # >10% difference
    }

# Example: rodents vs. primates (rodents evolve faster)
# Outgroup: fish
result = relative_rate_test(
    d_AC=0.42,   # mouse–fish distance
    d_BC=0.38,   # human–fish distance
    d_AB=0.18,   # mouse–human distance
)
print("Relative rate test: mouse (A) vs. human (B), fish (C = outgroup)")
for k, v in result.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print("\nMouse branch is longer → mouse lineage evolves faster → clock violated")

In [ ]:
# Cell 6.3 — Confidence interval on divergence time (Poisson sampling)
def clock_confidence_interval(d: float, r: float, L: int,
                               alpha: float = 0.05) -> tuple:
    """
    Approximate 95% CI on divergence time from Poisson uncertainty in observed distance.
    L: alignment length in base pairs
    """
    t_est = d / (2 * r)
    # Var(d) ≈ d*(1-d)/L for observed proportion; here use simplified σ_d = sqrt(d/L)
    sigma_d = np.sqrt(d * (1 - d) / L)
    from scipy import stats
    z = stats.norm.ppf(1 - alpha / 2)
    t_lo = (d - z * sigma_d) / (2 * r)
    t_hi = (d + z * sigma_d) / (2 * r)
    return t_lo, t_est, t_hi

print("Effect of alignment length on divergence time confidence interval:")
print(f"  (Human–chimp cytb: d={d_hc}, r={r_cytb:.6f})")
print(f"  {'Alignment length':>20} {'Lower 95%':>12} {'Estimate':>12} {'Upper 95%':>12} {'Width (My)':>12}")
for L in [500, 1000, 5000, 10000, 100000]:
    lo, mid, hi = clock_confidence_interval(d_hc, r_cytb, L)
    print(f"  {L:>20,} {lo:>12.2f} {mid:>12.2f} {hi:>12.2f} {hi-lo:>12.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Molecular clock: distance vs. time + rate variation across taxa
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Molecular clock — distance accumulates linearly with time
ax = axes[0]
t_range = np.linspace(0, 60, 200)
r_primate = r_cytb
r_rodent  = r_cytb * 2.0  # rodents ~2× faster
r_slow    = r_cytb * 0.5  # slow-evolving lineage

ax.plot(t_range, 2 * r_primate * t_range, color='steelblue', lw=2, label='Primate rate')
ax.plot(t_range, 2 * r_rodent  * t_range, color='tomato',    lw=2, label='Rodent rate (2×)')
ax.plot(t_range, 2 * r_slow    * t_range, color='seagreen',  lw=2, label='Slow lineage (0.5×)')

# Fossil calibration point
ax.scatter([t_fossil_hc_My], [d_hc], color='black', s=60, zorder=5)
ax.annotate('Human-chimp\nfossil calibration', (t_fossil_hc_My, d_hc),
            xytext=(10, 0.015), arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=8)
ax.set_xlabel('Time since divergence (My)')
ax.set_ylabel('JC genetic distance')
ax.set_title('Molecular clock:\nrate variation violates strict clock assumption')
ax.legend(fontsize=8)

# Panel 2: CI width vs. alignment length
ax = axes[1]
L_values = np.logspace(2, 6, 50).astype(int)
widths = []
for L in L_values:
    lo, mid, hi = clock_confidence_interval(d_hc, r_cytb, L)
    widths.append(hi - lo)

ax.loglog(L_values, widths, color='steelblue', lw=2)
ax.axhline(1.0, color='tomato', ls='--', lw=1, label='±0.5 My precision')
ax.set_xlabel('Alignment length (bp)')
ax.set_ylabel('95% CI width (My)')
ax.set_title('Longer alignments → narrower\ndivergence time estimates')
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `estimate_rate(d_calibrated, t_fossil)` and `molecular_clock_date(d, r)`
   from scratch. Using cytochrome b, estimate when the human–orangutan split occurred
   (d ≈ 0.035; literature value: ~14 My).
2. The relative rate test shows mouse evolves 2× faster than human (relative to a
   fish outgroup). If you use the primate rate to date the rodent–primate split from
   a rodent–fish distance, will you overestimate or underestimate the age? By how much?
3. BEAST2 estimates a 95% HPD (highest posterior density) of [8.2, 11.4] My for the
   human–gorilla split. What is an HPD, and how does it differ from a frequentist
   confidence interval?
4. Ancient DNA sequences from a Neanderthal dated at 50,000 years ago show
   d ≈ 0.00182 from modern humans. Using μ = 1.2×10⁻⁸ per site per year, what
   does this distance imply for the human-Neanderthal split time?

---
## Quiz — Active Recall

1. State the molecular clock equation. What are its three variables?
2. What is the factor of 2 in t = d/(2r)?
3. What is the relative rate test? What does a positive result imply?
4. Name two reasons the strict clock fails across mammalian lineages.
5. What is BEAST2 and what does it output?

---
## Reflection

**Date completed:** ____________________

> *[A paper claims human and Neanderthal diverged 600,000 years ago based on a
> mitochondrial clock. A colleague says the nuclear clock gives 800,000 years.
> Why might these differ, and which should you trust more?]*

---
*Next: `09_evolutionary_game_theory.ipynb`*